# Plant Doctor — Part 2: Training, Testing & Evaluating a Model

*A follow-up to the **Plant Doctor Challenge** game.*

In Part 1 you played the role of a single neuron: you read four measurements from a plant
sample, computed `z = w·x + b`, squashed it with the sigmoid into a disease probability,
and the **Coach** corrected the weights once. That showed you **how one neuron learns from one sample**.

A real model is never trained on one sample, and it is never judged on samples it has
already seen. In this notebook you'll learn the four ideas that turn "one neuron, one
correction" into "a model you can actually trust":

1. **Training set vs. Testing set** — why we *hide* some data from the model.
2. **Epochs** — what it means to loop the Coach's correction over the whole dataset, many times.
3. **Evaluation metrics** — **Accuracy, Precision, Recall, F1** — four different ways to score
   how good the model is, and why accuracy alone can lie to you.
4. **The decision threshold** — how moving the 0.5 cutoff trades one kind of mistake for another.

The model is the **same single sigmoid neuron** from the game — nothing more complicated.
We just give it more data and score it honestly.

> **How to use this notebook:** Read the explanation above each cell, then run the cell
> (`Shift+Enter`). Cells marked **🎯 EXERCISE** ask *you* to change something and predict
> what will happen before you run it. An **Answer Key** is at the very bottom.


## Cell 1 — Setup

We only need three standard libraries:
- **numpy** for the arithmetic (weighted sums, the sigmoid),
- **matplotlib** to draw learning curves,
- **pandas** just to print tidy tables.

Run this first. Nothing to change here.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)   # makes the "random" data the same every time we run -> reproducible

FEATURE_NAMES = ["SNP Diversity", "Pathogen Load", "Leaf Color", "Root Biomass"]

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

print("Setup complete. Same 4 features as the game:")
print(FEATURE_NAMES)


## Cell 2 — Build a bigger field dataset

In the game there were only **4** samples. You can't learn much — or test anything — with 4
plants. Here we generate **200** lab-confirmed samples that follow the same biology:

- **Higher pathogen load → more likely Sick.**
- **Higher leaf color, root biomass, and SNP diversity → more likely Healthy.**

We create each plant's four measurements at random, combine them with a "true" biological
rule, add a little noise (nature is messy — two plants with identical readings don't always
have identical outcomes), and record the lab-confirmed label:

- **label = 1 → Healthy**
- **label = 0 → Sick**

This is exactly the labelling convention from the game. `X` holds the measurements (200 rows ×
4 features); `y` holds the 200 confirmed diagnoses. Notice from the printout that the classes are
**imbalanced** — most field plants are Healthy — which becomes important when we score the model.


In [ ]:
N = 200

# Four measurements per plant, each normalized to 0-1 (like the game's samples)
snp   = np.random.rand(N)
path  = np.random.rand(N)
leaf  = np.random.rand(N)
root  = np.random.rand(N)
X = np.column_stack([snp, path, leaf, root])

# The hidden "true biology": a plant tends to be Healthy when diversity/greenness/roots are
# high and pathogen load is low. We add noise so the rule isn't perfectly clean.
true_score = (1.2*snp - 2.2*path + 1.3*leaf + 1.0*root
              + np.random.normal(0, 0.35, size=N))  # biological noise (nature is messy)
# Real field surveys are imbalanced: most sampled plants are Healthy. We keep that here
# on purpose, because it's exactly the situation where "accuracy" becomes misleading (Cell 8).
y = (true_score > 0).astype(int)                     # 1 = Healthy, 0 = Sick

df = pd.DataFrame(X, columns=FEATURE_NAMES)
df["Diagnosis"] = np.where(y == 1, "Healthy (1)", "Sick (0)")
print(f"Generated {N} samples.")
print(f"Healthy: {(y==1).sum()}   |   Sick: {(y==0).sum()}")
df.head(8)


## Cell 3 — Concept 1: Training set vs. Testing set

**The single most important idea in this notebook.**

Imagine a student who "studies" by memorizing the answer key to last year's exam. They'll ace
*that* exam — and fail the moment they see a new question. A model that is scored on the same
plants it learned from can *memorize* instead of *learn*, and you'd never know it was useless
until it hit real field samples.

So we **split** the 200 samples into two piles:

- **Training set (≈70%)** — the plants the model is *allowed to learn from*. The Coach adjusts
  the weights using only these.
- **Testing set (≈30%)** — plants we **lock in a drawer** and never let the model see during
  training. At the end we bring them out to ask: *"Here are plants you've never seen — can you
  diagnose them?"* That score is the one we trust.

We shuffle first (so the split isn't accidentally biased) and then cut. Notice the test plants
never touch the training loop in Cell 4.


In [ ]:
# Shuffle the row order so the split is unbiased
perm = np.random.permutation(N)
X, y = X[perm], y[perm]

split = int(0.70 * N)          # 70% train, 30% test  <-- you'll change this in an exercise
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training set: {len(X_train)} plants  (the model learns from these)")
print(f"Testing set : {len(X_test)} plants  (locked away until evaluation)")


## Cell 4 — Concept 2: Epochs (training the neuron)

In the game, the Coach corrected the weights **once**, for **one** plant. Training a real model
just repeats that same correction:

- For **every plant in the training set**, compute the prediction `σ(z)`, then nudge the weights
  with the *exact rule from the game*:

  `w ← w + η · (True − σ(z)) · x`   and   `b ← b + η · (True − σ(z))`

- Doing that once for **all** training plants is called **one epoch**.
- We repeat for many epochs. Each epoch, the weights get a little better, so the **loss**
  (how wrong the model is) goes down and **accuracy** goes up — until it levels off (converges).

**Reading that update rule.** For each weight you always *add* `η · (True − σ(z)) · x`.
Three pieces do three separate jobs:

| Piece | What it is | What it does |
|---|---|---|
| `(True − σ(z))` | the **error**: lab answer minus the model's guess | sets the **direction** (and shrinks toward 0 once the guess is right) |
| `x` | the **measurement** feeding that weight | assigns **blame** — a big feature gets a big nudge; a near-zero feature is barely touched |
| `η` | the **learning rate** | sets the **step size** |

You never have to decide "add or subtract" — the error's own sign does it for you:
- guess **too low** (said 0.3 Healthy, truth = Healthy 1): `1 − 0.3 = +0.7` → the weight goes **up**;
- guess **too high** (said 0.8 Healthy, truth = Sick 0): `0 − 0.8 = −0.8` → the weight goes **down**;
- guess **already right** (0.99 vs 1): `+0.01` → the weight barely moves.

*(Yes, it's a `+`, not the `−` you might expect from "gradient descent" — because the error term
`(True − σ(z))` already carries the sign. Same rule as the Part 1 game.)*

> **What this formula is called.** It's one step of **stochastic gradient descent (SGD)** on the
> cross-entropy loss — i.e. the standard training rule for **logistic regression** (a single sigmoid
> neuron). In classic neural-network terms it's known as the **delta rule** (or Widrow–Hoff / LMS
> rule), and the very same `(error) × (input)` calculation, run through a *multi-layer* network, is
> what **backpropagation** does.

**One step, concretely.** Suppose for one plant the root-biomass reading is `x = 0.8`, its current
weight is `w = 0`, the model guessed `σ(z) = 0.3`, the plant is truly Healthy (`True = 1`), and `η = 0.05`:

`error = 1 − 0.3 = +0.7`  →  `step = η · error · x = 0.05 · 0.7 · 0.8 = +0.028`  →  `w = 0 + 0.028 = 0.028`

The weight ticked **up**, so next time this plant's high root biomass pushes `σ(z)` closer to 1 —
closer to the truth. Do that for every plant, every epoch, and the weights settle where the guesses
match the lab results. **That loop is exactly what "training" is.**

Think of an epoch as the Coach walking down the whole bench of confirmed samples once,
correcting after each; more epochs = more passes down the bench.

We start every weight at 0 (just like the game's reset) and record the training loss, training
accuracy, and — importantly — the **test** accuracy after each epoch, so we can watch learning happen.


In [ ]:
def cross_entropy(y_true, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)          # avoid log(0)
    return -np.mean(y_true*np.log(p) + (1-y_true)*np.log(1-p))

def accuracy(y_true, p):
    return np.mean((p >= 0.5).astype(int) == y_true)

# ---- knobs you'll experiment with in the exercises ----
EPOCHS = 60
LEARNING_RATE = 0.05
# -------------------------------------------------------

w = np.zeros(4)      # w1..w4, all start at 0 (like the game's reset)
b = 0.0              # bias starts at 0

history = {"epoch": [], "train_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    # ---- one epoch = one full pass over the TRAINING set ----
    for i in range(len(X_train)):
        x_i = X_train[i]
        p_i = sigmoid(np.dot(w, x_i) + b)     # forward pass (game Steps 1-3)
        error = y_train[i] - p_i              # "True - sigma(z)"  (game's Error)
        w += LEARNING_RATE * error * x_i      # gradient-descent update (game's Step 5)
        b += LEARNING_RATE * error

    # ---- after the epoch, measure progress on train AND test ----
    p_train = sigmoid(X_train @ w + b)
    p_test  = sigmoid(X_test  @ w + b)
    history["epoch"].append(epoch)
    history["train_loss"].append(cross_entropy(y_train, p_train))
    history["train_acc"].append(accuracy(y_train, p_train))
    history["test_acc"].append(accuracy(y_test, p_test))

print("Training finished.")
print("Final weights:", np.round(w, 3), " bias:", round(b, 3))
print(f"Train accuracy: {history['train_acc'][-1]:.3f}   Test accuracy: {history['test_acc'][-1]:.3f}")


## Cell 5 — Watch the epochs work: learning curves

The numbers above are a snapshot; the **learning curve** is the movie. We plot, for each epoch:

- **Left:** training **loss** — should fall and flatten as the neuron stops being wrong.
- **Right:** **accuracy** on the training set (blue) and the hidden test set (orange).

Look for the point where the curves **flatten** — that's roughly how many epochs this problem
actually needs. Running many more epochs after that buys you almost nothing. Also notice the
train and test curves sit close together here: the model is *generalizing*, not memorizing.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

ax1.plot(history["epoch"], history["train_loss"], color="#c0392b")
ax1.set_title("Training loss vs. epoch")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Cross-entropy loss"); ax1.grid(alpha=0.3)

ax2.plot(history["epoch"], history["train_acc"], label="Train accuracy", color="#2E5B8A")
ax2.plot(history["epoch"], history["test_acc"],  label="Test accuracy",  color="#C9971F")
ax2.set_title("Accuracy vs. epoch")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.set_ylim(0, 1.02)
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()


## Aside — What about regularization (weight decay)?

You may have heard that real models use **regularization**. Here's where it fits — and why this
notebook deliberately doesn't use it.

Regularization fights **overfitting**: a model *memorizing* its training data instead of learning
the real pattern. The most common form, **L2 regularization** (also called **weight decay**), adds
one term to the Cell 4 update rule that gently pulls every weight back toward zero on each step:

`w ← w + η · [ (True − σ(z)) · x  −  λ · w ]`

The extra **`− λ · w`** term (`λ` = regularization strength) discourages the weights from growing
large and over-committing to any single feature — bigger `λ` = stronger pull toward simpler, smaller
weights.

**Why this notebook leaves it out.** Our model is a single sigmoid neuron with only **5 numbers to
learn** (4 weights + 1 bias), trained on 200 plants. It's far too simple to overfit — and you can
*see* that in the learning curves above: the **test** accuracy tracks the **train** accuracy instead
of lagging behind it. When train and test stay close like this, there's **no overfitting gap for
regularization to close**, so switching it on would change almost nothing here.

You reach for regularization when the picture above looks *different* — train accuracy climbing while
**test** accuracy *drops*, a widening gap between the two curves. That happens with many features,
very few training samples, or big multi-layer networks. Spotting that gap is exactly what the
**train/test split (Cell 3)** and these **learning curves (Cell 5)** are designed to let you do.


## Cell 6 — Concept 3: Evaluation starts with the confusion matrix

Now we grade the model **only on the locked-away test set**. Every prediction falls into one of
four buckets. Because we're screening for *disease*, we call **"Sick" the positive class** (the
event we're trying to catch):

| | Model says **Sick** | Model says **Healthy** |
|---|---|---|
| **Truly Sick**   | ✅ True Positive (TP) — caught the disease | ❌ False Negative (FN) — **missed a sick plant** |
| **Truly Healthy**| ❌ False Positive (FP) — **false alarm** | ✅ True Negative (TN) — correctly cleared |

The two mistakes are *not* equally bad, and they trade off against each other:
- a **False Negative** = a diseased plant sent back to the field to infect its neighbors;
- a **False Positive** = a healthy plant needlessly quarantined or sprayed.

The cell below counts the four buckets on the test set, prints them as a table, and draws them
as a **colour-coded confusion-matrix heatmap** (green = correct, red = the two kinds of mistake).
Every metric in the next cell is just a different way of combining these four counts.


In [ ]:
# Predictions on the hidden test set
p_test = sigmoid(X_test @ w + b)
pred_healthy = (p_test >= 0.5).astype(int)      # 1 = model says Healthy

# Convert to the "Sick = positive" view for disease screening
true_sick = (y_test == 0).astype(int)           # 1 = truly Sick
pred_sick = (pred_healthy == 0).astype(int)     # 1 = model says Sick

TP = int(np.sum((pred_sick == 1) & (true_sick == 1)))
FP = int(np.sum((pred_sick == 1) & (true_sick == 0)))
FN = int(np.sum((pred_sick == 0) & (true_sick == 1)))
TN = int(np.sum((pred_sick == 0) & (true_sick == 0)))

cm = pd.DataFrame([[TP, FN], [FP, TN]],
                  index=["Truly Sick", "Truly Healthy"],
                  columns=["Model: Sick", "Model: Healthy"])
print("Confusion matrix (positive class = Sick):")
print(cm)
print(f"\nTP={TP}  FP={FP}  FN={FN}  TN={TN}   (total test plants = {len(y_test)})")

# ---- draw it as a heatmap so the four buckets are easy to read ----
counts = np.array([[TP, FN], [FP, TN]])
labels = np.array([[f"TP\n{TP}", f"FN\n{FN}"], [f"FP\n{FP}", f"TN\n{TN}"]])
# green for correct (TP, TN on the diagonal), red for mistakes (FN, FP)
colors = np.array([[0, 1], [1, 0]])   # 0 -> green cell, 1 -> red cell

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(colors, cmap=plt.cm.RdYlGn_r, vmin=0, vmax=1, alpha=0.35)
for i in range(2):
    for j in range(2):
        ax.text(j, i, labels[i, j], ha="center", va="center", fontsize=14, fontweight="bold")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Model: Sick", "Model: Healthy"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Truly Sick", "Truly Healthy"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion matrix on the hidden test set\n(green = correct, red = mistakes)")
plt.tight_layout(); plt.show()


## Cell 7 — Concept 3 (cont.): Accuracy, Precision, Recall, F1

Four scores, four different questions. Learn what each one *asks*:

**Accuracy — "Overall, how often am I right?"**
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
Simple, but **dangerous when classes are imbalanced.** If 95% of plants are Healthy, a lazy model
that says "Healthy" every time scores 95% accuracy while catching **zero** sick plants.

**Precision — "When I raise the alarm, how often am I right?"**
$$\text{Precision} = \frac{TP}{TP + FP}$$
High precision = few false alarms. *You're not wasting fungicide on healthy plants.*

**Recall (Sensitivity) — "Of all the truly sick plants, how many did I catch?"**
$$\text{Recall} = \frac{TP}{TP + FN}$$
High recall = few misses. *No diseased plant slips back into the field.*

**F1 — one number that balances precision and recall** (their *harmonic* mean, which punishes
having one of them low):
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

We compute all four by hand from TP/FP/FN/TN, then double-check against scikit-learn so you can
see the formulas and the library agree.


In [ ]:
acc       = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) else 0.0
recall    = TP / (TP + FN) if (TP + FN) else 0.0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

print("Computed by hand (positive class = Sick):")
print(f"  Accuracy : {acc:.3f}   -> overall fraction correct")
print(f"  Precision: {precision:.3f}   -> of alarms raised, fraction truly sick")
print(f"  Recall   : {recall:.3f}   -> of truly sick plants, fraction caught")
print(f"  F1 score : {f1:.3f}   -> balance of precision & recall")

# --- Cross-check with scikit-learn (optional but reassuring) ---
try:
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    print("\nscikit-learn agrees:")
    print(f"  Accuracy : {accuracy_score(true_sick, pred_sick):.3f}")
    print(f"  Precision: {precision_score(true_sick, pred_sick, zero_division=0):.3f}")
    print(f"  Recall   : {recall_score(true_sick, pred_sick, zero_division=0):.3f}")
    print(f"  F1 score : {f1_score(true_sick, pred_sick, zero_division=0):.3f}")
except ImportError:
    print("\n(scikit-learn not installed - the hand-computed values above are what matter.)")


## Cell 8 — Why accuracy can lie (the "always Healthy" model)

Let's prove the warning from Cell 7. We build a **fake model that ignores every measurement and
always predicts "Healthy."** Because most plants really are Healthy, it scores a deceptively
decent accuracy **for free** — while catching **zero** sick plants.

This is the single most important habit in model evaluation: **never report accuracy alone.**
A screening tool that never catches a sick plant is worthless no matter how high its accuracy looks.


In [ ]:
# A "model" that always says Healthy -> it flags NOBODY as sick
lazy_pred_sick = np.zeros_like(true_sick)   # predicts Sick for 0 plants

lazy_TP = int(np.sum((lazy_pred_sick == 1) & (true_sick == 1)))   # = 0
lazy_FN = int(np.sum((lazy_pred_sick == 0) & (true_sick == 1)))
lazy_TN = int(np.sum((lazy_pred_sick == 0) & (true_sick == 0)))

lazy_acc    = (lazy_TP + lazy_TN) / len(true_sick)
lazy_recall = lazy_TP / (lazy_TP + lazy_FN) if (lazy_TP + lazy_FN) else 0.0

print("The lazy 'always Healthy' model:")
print(f"  Accuracy: {lazy_acc:.3f}   <- sounds okay-ish, and it's FREE: it just always guesses 'Healthy'")
print(f"  Recall  : {lazy_recall:.3f}   <- but it caught {lazy_TP} of the {int(true_sick.sum())} sick plants. USELESS.")
print(f"\nA {lazy_acc:.0%}-accurate model that catches zero disease is worthless for screening -")
print("and accuracy alone would never have warned you. Always report recall / precision / F1 too.")


---
# 🎯 Exercises

Now it's your turn. For each one, **predict what will happen before you run it**, then check.
Solutions are in the Answer Key at the end.


## 🎯 Exercise 1 — How big should the test set be?

In **Cell 3** we used a 70/30 split. Change the split fraction below and re-run this cell,
then look at how many plants end up in each pile.

**Before running, predict:** if you train on only 10% of the data (`0.10`), do you expect the
test accuracy to go **up** or **down**? Why? Try `0.10`, `0.50`, and `0.90`.


In [ ]:
SPLIT_FRACTION = 0.70   # <-- CHANGE ME (try 0.10, 0.50, 0.90)

split_ex = int(SPLIT_FRACTION * N)
Xtr, Xte = X[:split_ex], X[split_ex:]
ytr, yte = y[:split_ex], y[split_ex:]

# quick train with the same rule as Cell 4
w_ex, b_ex = np.zeros(4), 0.0
for _ in range(60):
    for i in range(len(Xtr)):
        p = sigmoid(np.dot(w_ex, Xtr[i]) + b_ex)
        err = ytr[i] - p
        w_ex += 0.05 * err * Xtr[i]; b_ex += 0.05 * err

test_acc_ex = accuracy(yte, sigmoid(Xte @ w_ex + b_ex))
print(f"Split = {SPLIT_FRACTION:.0%} train / {1-SPLIT_FRACTION:.0%} test")
print(f"  Train plants: {len(Xtr)}   Test plants: {len(Xte)}")
print(f"  Test accuracy: {test_acc_ex:.3f}")


## 🎯 Exercise 2 — How many epochs are enough?

Go back to **Cell 4**, change `EPOCHS` to **5**, re-run Cell 4 and Cell 5, and read the final
train/test accuracy. Then try `EPOCHS = 300`.

**Predict first:** Will 5 epochs be enough for the loss curve to flatten? Will 300 epochs make
the **test** accuracy dramatically better than 60 did, or has it already converged?

Use the cell below to compare a few epoch counts at once and see where the gains stop.


In [ ]:
for ep in [1, 5, 20, 60, 300]:
    w_e, b_e = np.zeros(4), 0.0
    for _ in range(ep):
        for i in range(len(X_train)):
            p = sigmoid(np.dot(w_e, X_train[i]) + b_e)
            err = y_train[i] - p
            w_e += 0.05 * err * X_train[i]; b_e += 0.05 * err
    tr = accuracy(y_train, sigmoid(X_train @ w_e + b_e))
    te = accuracy(y_test,  sigmoid(X_test  @ w_e + b_e))
    print(f"epochs={ep:4d}   train acc={tr:.3f}   test acc={te:.3f}")


## 🎯 Exercise 3 — Compute the metrics by hand

A classmate's model produced this confusion matrix on 100 test plants (positive class = **Sick**):

| | Model: Sick | Model: Healthy |
|---|---|---|
| **Truly Sick**    | TP = 30 | FN = 20 |
| **Truly Healthy** | FP = 10 | TN = 40 |

**Fill in the four formulas** below (replace each `FILL_IN`) and run to check yourself.
Then answer in a sentence: *is this model better at avoiding false alarms, or at catching disease?*


In [ ]:
TP_ex, FN_ex, FP_ex, TN_ex = 30, 20, 10, 40

accuracy_ex  = 0    # TODO: (TP + TN) / total
precision_ex = 0    # TODO: TP / (TP + FP)
recall_ex    = 0   # TODO: TP / (TP + FN)
f1_ex        = 0    # TODO: 2 * P * R / (P + R)

print(f"Accuracy : {accuracy_ex:.3f}")
print(f"Precision: {precision_ex:.3f}")
print(f"Recall   : {recall_ex:.3f}")
print(f"F1       : {f1_ex:.3f}")


## 🎯 Exercise 4 — Move the decision threshold (precision vs. recall)

So far a plant is called **Sick** when `σ(z) < 0.5`, i.e. `P(Healthy) < 0.5`. But **0.5 is a
choice, not a law.** If missing a sick plant is very costly, we can lower the bar for raising
the alarm — flag a plant as Sick even when we're only, say, 60% sure it's healthy
(`threshold = 0.60`). That catches more disease (**higher recall**) but raises more false alarms
(**lower precision**).

Run the sweep below and watch precision and recall move in **opposite** directions. Then answer:
*for a nursery that must not let ANY disease escape, which threshold would you pick?*


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

print(f"{'threshold':>10} {'precision':>10} {'recall':>8} {'F1':>7}")
for thr in [0.30, 0.40, 0.50, 0.60, 0.70]:
    pred_sick_thr = (sigmoid(X_test @ w + b) < thr).astype(int)   # call Sick below the threshold
    P = precision_score(true_sick, pred_sick_thr, zero_division=0)
    R = recall_score(true_sick, pred_sick_thr, zero_division=0)
    F = f1_score(true_sick, pred_sick_thr, zero_division=0)
    print(f"{thr:>10.2f} {P:>10.3f} {R:>8.3f} {F:>7.3f}")


> ### ⚠️ One honest caveat: don't tune on the test set
>
> Notice what you just did in Exercises 2 and 4: you picked the **number of epochs** and the
> **decision threshold** by looking at **test-set** accuracy. In a real project that's a subtle form of
> cheating — every time you adjust a knob based on the test set, you leak a little information about it
> into your choice, so your final test score ends up **optimistic** (better than the model will do on
> truly new data).
>
> The professional fix is a **three-way split**: **train / validation / test**. You *train* on the
> training set, *tune* every knob (epochs, threshold, layer size…) against the **validation** set, and
> **touch the test set only once, at the very end**, for a single honest estimate. We used just
> train/test here to keep the focus on the core ideas — but now you know the habit: **the test set is a
> one-time final exam, not something you optimize against.**


---
# ✅ Answer Key

**Exercise 1 — test-set size.** With only 10% for training the model sees too few examples to
pin down good weights, so **test accuracy usually drops and gets noisier**. As you give it more
training data (50%, 70%, 90%) accuracy generally improves and stabilizes — but at 90% train / 10%
test the *test* score becomes unreliable simply because it's measured on very few plants (a few
lucky or unlucky cases swing it a lot). The usual compromise (≈70–80% train) balances *enough data
to learn* against *enough held-out data to measure honestly.*

**Exercise 2 — epochs.** With 1–5 epochs the model clearly **underfits** less than 0.85. By 20 60 epochs accuracy has largely
**converged** (0.88–0.92), and pushing on to 300 epochs adds only a small further gain. Key idea:
**more epochs help only until convergence**; past that you're mostly spending compute for nothing
(and on harder models, risking overfitting).

**Exercise 3 — metrics by hand.**
- Accuracy = (30 + 40) / 100 = **0.70**
- Precision = 30 / (30 + 10) = **0.75**
- Recall = 30 / (30 + 20) = **0.60**
- F1 = 2 · (0.75 · 0.60) / (0.75 + 0.60) = 0.90 / 1.35 = **0.667**

Precision (0.75) > Recall (0.60), so this model is **better at avoiding false alarms than at
catching disease** — when it says "Sick" it's usually right, but it still **misses 20 of the 50
truly sick plants**. For a screening tool that's a problem.

**Exercise 4 — threshold.** As the threshold rises (0.30 → 0.70) you flag more plants as Sick, so
**recall goes up while precision goes down** — the classic trade-off. A nursery that must let **no**
disease escape cares about **recall** and would pick a **higher threshold** (e.g. 0.60–0.70),
accepting more false alarms (extra inspections) as the price of missing fewer sick plants. F1
balances the two — here it stays highest across the middle-to-higher thresholds (~0.5–0.7) — which
is why it's a good default single number when both kinds of error matter.


---
### Recap — Part 2 (Training, Testing & Evaluation)
- **Why** we hold out a **test set**, and how to split data into train/test.
- What an **epoch** is, and how to read a **learning curve** to judge convergence.
- Build a **confusion matrix** and compute **accuracy, precision, recall, and F1** by hand.
- Why **accuracy alone misleads**, and how the **decision threshold** trades precision vs. recall.

*everything so far used a *single neuron*.
Lets practice with a simulation of many neuron that will make a real network.
go here: https://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.86495&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false
